# ERA5 Pipeline — Annotated Walkthrough

A step-by-step learning reference for the ERA5 data pipeline.
Covers real ARCO ERA5 from Google Cloud → local zarr → regrid → Dask stats → PyTorch DataLoader.

**Variables (VPD drivers):**

| Variable | Level | Physical role |
|---|---|---|
| `2m_temperature` | Surface | → e_s(T), saturation vapor pressure |
| `2m_dewpoint_temperature` | Surface | → e_a; VPD = e_s(T) - e_s(Td) |
| `10m_u/v_component_of_wind` | Surface | Surface mixing, evaporative demand |
| `temperature` → `temperature_850` | 850 hPa | Air mass advection → drives T2m |
| `specific_humidity` → `specific_humidity_850` | 850 hPa | Moisture advection → drives Td2m |
| `geopotential` → `geopotential_500` | 500 hPa | Synoptic pattern: ridge → hot/dry, trough → cool/moist |

**Source:** `gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3`

In [35]:
from __future__ import annotations
import os, shutil, time, warnings
from typing import Sequence

import numpy as np
import torch
import xarray as xr
import dask
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings('ignore', category=UserWarning)

## Config

Keep the first pass small — so fits comfortably in local dev memory.

In [36]:
ARCO_ERA5_PATH = 'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3'

# Surface variables — no pressure level, shape (time, lat, lon)
SURFACE_VARS = [
    '2m_temperature',            # T2m → e_s(T)
    '2m_dewpoint_temperature',   # Td2m → e_a;  VPD = e_s(T2m) - e_s(Td2m)
    '10m_u_component_of_wind',   # surface zonal wind
    '10m_v_component_of_wind',   # surface meridional wind
]

# Pressure-level variables: ERA5 name → level to extract (hPa)
# Selecting one level converts (time, level, lat, lon) → (time, lat, lon)
LEVEL_VARS = {
    'temperature': 850,       # T850 — warm/cold air mass advection → drives T2m
    'specific_humidity': 850, # q850 — moisture advection → drives Td2m
    'geopotential': 500,      # Z500 — synoptic pattern; ridge = hot/dry, trough = cool/moist
}

# ARCO download list (multi-level vars before extraction)
VARIABLES = SURFACE_VARS + list(LEVEL_VARS.keys())

# Final 2D variable names written to zarr and used by ERA5Dataset
ML_VARS = SURFACE_VARS + [f'{v}_{lev}' for v, lev in LEVEL_VARS.items()]
# → ['2m_temperature', '2m_dewpoint_temperature', '10m_u_component_of_wind',
#    '10m_v_component_of_wind', 'temperature_850', 'specific_humidity_850', 'geopotential_500']

TIME_START = '2020-06-01'
TIME_END   = '2020-06-05'   # 4 days = 96 hourly timesteps

# Northeastern MN / Duluth–Manitou area (ERA5 uses 0–360 longitude)
LAT_MAX = 48.0
LAT_MIN = 45.0
LON_MIN = 263.0   # ~97°W
LON_MAX = 271.0   # ~89°W

LOCAL_ZARR_PATH   = '../data/era5_subset.zarr'
LOCAL_REGRID_PATH = '../data/era5_subset_1deg.zarr'

os.makedirs('../data', exist_ok=True)

## Dask Setup

On M2/32GB, the **threaded scheduler** is still preferred over `LocalCluster`:
- No subprocess spawning overhead
- No port conflicts
- Threads release the GIL for numpy/zarr I/O so parallelism works
- With 32GB headroom, 8 workers matches M2 performance cores

`'synchronous'` = single thread (good for debugging)  
`'threads'` = thread pool (good for I/O-heavy workloads like zarr reads)

`LocalCluster` is now viable on 32GB if you want the task-graph dashboard at `localhost:8787`, but adds ~500MB overhead and isn't needed here.



### Dask has three built-in schedulers that don't require any setup:
### synchronous — 
runs tasks one at a time in the main thread. No parallelism at all. Useful when debugging because tracebacks are clean and you can step through with a debugger. Use this when something is going wrong and you want to see exactly where.
### threads — 
spawns a thread pool. This is what the note is recommending. For zarr I/O, threads work well because the actual read operations (decompressing chunks from GCS, numpy operations) release Python's GIL, so multiple threads can genuinely run in parallel even though Python normally serializes everything. On your M2 with 8 performance cores, 8 threads means 8 chunks can decompress simultaneously.
### processes — 
spawns separate Python processes. Bypasses the GIL entirely but has high overhead: each process needs its own Python interpreter and memory space, can't share objects directly, and serialization cost is high. Overkill for I/O-bound workloads.

In [37]:
dask.config.set(scheduler='threads', num_workers=8)
print('Dask scheduler: threaded (8 workers, M2/32GB)')

Dask scheduler: threaded (8 workers, M2/32GB)


## Step 1: Open & Subset ARCO ERA5

`xr.open_zarr` is **lazy** — nothing is downloaded until `.compute()` is called.

Key points:
- `chunks={'time': 1}` tells Dask to treat each timestep as its own task
- ERA5 latitude is stored **descending** (90 → -90), so `slice(lat_max, lat_min)`
- Rename `latitude/longitude` → `lat/lon` for xESMF compatibility
- **Level extraction:** pressure-level vars (`temperature`, `specific_humidity`, `geopotential`) are stored as (time, level, lat, lon). We `.sel(level=N)` to extract a single level and drop the dimension → all vars end up as (time, lat, lon) so ERA5Dataset can stack them

In [38]:
print('Opening ARCO ERA5 (lazy — nothing downloaded yet)...')

ds = xr.open_zarr(
    ARCO_ERA5_PATH,
    consolidated=True,
    storage_options={'token': 'anon'},
    chunks={'time': 1},
)

print(f'Remote vars available: {list(ds.data_vars)[:8]} ...')

# Subset: variables + region + time window
subset = ds[VARIABLES].sel(
    time=slice(TIME_START, TIME_END),
    latitude=slice(LAT_MAX, LAT_MIN),   # descending → slice(max, min)
    longitude=slice(LON_MIN, LON_MAX),
)

subset = subset.rename({'latitude': 'lat', 'longitude': 'lon'})

# Extract specific pressure levels → rename and drop level dimension
# Before: geopotential shape (time, level, lat, lon)
# After:  geopotential_500 shape (time, lat, lon)  — stackable with surface vars
for var, level in LEVEL_VARS.items():
    subset[f'{var}_{level}'] = subset[var].sel(level=level, drop=True)
subset = subset.drop_vars(list(LEVEL_VARS.keys()))

print(f'Subset dims: {dict(subset.sizes)}')
print(f'Variables: {list(subset.data_vars)}')

# Rough memory estimate
n_cells = subset.sizes['time'] * subset.sizes['lat'] * subset.sizes['lon']
mb = n_cells * len(subset.data_vars) * 4 / 1e6
print(f'Estimated full load size: {mb:.0f} MB across {len(subset.data_vars)} variables')

subset

Opening ARCO ERA5 (lazy — nothing downloaded yet)...
Remote vars available: ['100m_u_component_of_wind', '100m_v_component_of_wind', '10m_u_component_of_neutral_wind', '10m_u_component_of_wind', '10m_v_component_of_neutral_wind', '10m_v_component_of_wind', '10m_wind_gust_since_previous_post_processing', '2m_dewpoint_temperature'] ...
Subset dims: {'time': 120, 'lat': 13, 'lon': 33, 'level': 37}
Variables: ['2m_temperature', '2m_dewpoint_temperature', '10m_u_component_of_wind', '10m_v_component_of_wind', 'temperature_850', 'specific_humidity_850', 'geopotential_500']
Estimated full load size: 1 MB across 7 variables


<xarray.Dataset> Size: 1MB
Dimensions:                  (time: 120, lat: 13, lon: 33, level: 37)
Coordinates:
  * lat                      (lat) float32 52B 48.0 47.75 47.5 ... 45.25 45.0
  * level                    (level) int64 296B 1 2 3 5 7 ... 925 950 975 1000
  * lon                      (lon) float32 132B 263.0 263.2 ... 270.8 271.0
  * time                     (time) datetime64[ns] 960B 2020-06-01 ... 2020-0...
Data variables:
    2m_temperature           (time, lat, lon) float32 206kB dask.array<chunksize=(1, 13, 33), meta=np.ndarray>
    2m_dewpoint_temperature  (time, lat, lon) float32 206kB dask.array<chunksize=(1, 13, 33), meta=np.ndarray>
    10m_u_component_of_wind  (time, lat, lon) float32 206kB dask.array<chunksize=(1, 13, 33), meta=np.ndarray>
    10m_v_component_of_wind  (time, lat, lon) float32 206kB dask.array<chunksize=(1, 13, 33), meta=np.ndarray>
    temperature_850          (time, lat, lon) float32 206kB dask.array<chunksize=(1, 13, 33), meta=np.ndarray>
    specific_humidity_850    (time, lat, lon) float32 206kB dask.array<chunksize=(1, 13, 33), meta=np.ndarray>
    geopotential_500         (time, lat, lon) float32 206kB dask.array<chunksize=(1, 13, 33), meta=np.ndarray>
Attributes:
    valid_time_start:       1940-01-01
    last_updated:           2026-03-15 02:17:18.543343+00:00
    valid_time_stop:        2025-11-30
    valid_time_stop_era5t:  2026-03-09

## Step 2: Write Local Zarr

### Chunking strategy: `{time: 1, lat: -1, lon: -1}`

| Access pattern | Optimal chunks |
|---|---|
| ML training (sequential frames) | `{time: 1, lat: -1, lon: -1}` |
| Time-series analysis | `{time: -1, lat: 10, lon: 10}` |
| Spatial statistics | `{time: 24, lat: -1, lon: -1}` |

With `time: 1`, each chunk is one full spatial snapshot (~0.1 MB for CONUS at 0.25°).
ML DataLoader loads frames sequentially → one chunk per `__getitem__` call.

**Important:** `drop_encoding()` clears inherited remote encodings from ARCO ERA5.
Without this, zarr v3 codec format mismatches cause write errors.

In [39]:
ML_CHUNKS = {'time': 1, 'lat': -1, 'lon': -1}  # -1 = full dimension

# Clear inherited remote encodings — required to avoid zarr v3 codec conflicts
subset_chunked = subset.drop_encoding().chunk(ML_CHUNKS)
for v in list(subset_chunked.data_vars) + list(subset_chunked.coords):
    subset_chunked[v].encoding = {}

if os.path.exists(LOCAL_ZARR_PATH):
    shutil.rmtree(LOCAL_ZARR_PATH)

print(f'Chunk layout: {ML_CHUNKS}')
print(f'Writing to {LOCAL_ZARR_PATH} ...')

t0 = time.time()
subset_chunked.to_zarr(
    LOCAL_ZARR_PATH,
    mode='w',
    consolidated=True,   # writes consolidated metadata — faster remote reads
)
print(f'Done in {time.time() - t0:.1f}s')

# Re-open from local — now fully offline
ds_local = xr.open_zarr(LOCAL_ZARR_PATH, consolidated=True, chunks=ML_CHUNKS)
print(f'Local store dims: {dict(ds_local.sizes)}')
ds_local

Chunk layout: {'time': 1, 'lat': -1, 'lon': -1}
Writing to ../data/era5_subset.zarr ...
Done in 445.5s
Local store dims: {'time': 120, 'lat': 13, 'lon': 33, 'level': 37}


<xarray.Dataset> Size: 1MB
Dimensions:                  (time: 120, lat: 13, lon: 33, level: 37)
Coordinates:
  * lat                      (lat) float32 52B 48.0 47.75 47.5 ... 45.25 45.0
  * level                    (level) int64 296B 1 2 3 5 7 ... 925 950 975 1000
  * lon                      (lon) float32 132B 263.0 263.2 ... 270.8 271.0
  * time                     (time) datetime64[ns] 960B 2020-06-01 ... 2020-0...
Data variables:
    10m_u_component_of_wind  (time, lat, lon) float32 206kB dask.array<chunksize=(1, 13, 33), meta=np.ndarray>
    10m_v_component_of_wind  (time, lat, lon) float32 206kB dask.array<chunksize=(1, 13, 33), meta=np.ndarray>
    2m_dewpoint_temperature  (time, lat, lon) float32 206kB dask.array<chunksize=(1, 13, 33), meta=np.ndarray>
    2m_temperature           (time, lat, lon) float32 206kB dask.array<chunksize=(1, 13, 33), meta=np.ndarray>
    geopotential_500         (time, lat, lon) float32 206kB dask.array<chunksize=(1, 13, 33), meta=np.ndarray>
    specific_humidity_850    (time, lat, lon) float32 206kB dask.array<chunksize=(1, 13, 33), meta=np.ndarray>
    temperature_850          (time, lat, lon) float32 206kB dask.array<chunksize=(1, 13, 33), meta=np.ndarray>
Attributes:
    last_updated:           2026-03-15 02:17:18.543343+00:00
    valid_time_start:       1940-01-01
    valid_time_stop:        2025-11-30
    valid_time_stop_era5t:  2026-03-09

## Step 3: Regrid with xESMF

### Which method to use?

| Method | Use for | Preserves |
|---|---|---|
| `bilinear` | Temperature, wind, geopotential, soil moisture | Local point values |
| `conservative` | Precipitation, radiation, surface fluxes | Area integrals (mass/energy) |
| `nearest_s2d` | Categorical / mask fields | Discrete categories |

Conservative regridding is physically critical: bilinear-interpolated precipitation
can violate conservation of mass. ERA5 0.25° → WRF boundary conditions always
requires conservative for flux variables.

In [44]:
try:
    import xesmf as xe

    target_grid = xr.Dataset({
        'lat': (['lat'], np.arange(LAT_MIN, LAT_MAX + 1.0, 1.0)),
        'lon': (['lon'], np.arange(LON_MIN, LON_MAX + 1.0, 1.0)),
    })

    print(f'Source: {ds_local.sizes["lat"]} × {ds_local.sizes["lon"]} (0.25°)')
    print(f'Target: {len(target_grid.lat)} × {len(target_grid.lon)} (1.0°)')

    state_vars = ['2m_temperature', 'volumetric_soil_water_layer_1',
                  'leaf_area_index_high_vegetation', 't2m_celsius']
    state_vars = [v for v in state_vars if v in ds_local]

    regridder = xe.Regridder(
        ds_local[state_vars],
        target_grid,
        method='bilinear',
        periodic=False,
        reuse_weights=True,   # cache weights — saves time if called repeatedly
    )

    ds_regrid = regridder(ds_local[state_vars])
    print(f'Regridded dims: {dict(ds_regrid.sizes)}')

    if os.path.exists(LOCAL_REGRID_PATH):
        shutil.rmtree(LOCAL_REGRID_PATH)

    ds_regrid.chunk({'time': 1, 'lat': -1, 'lon': -1}).to_zarr(
        LOCAL_REGRID_PATH, mode='w', consolidated=True
    )
    print(f'Saved: {LOCAL_REGRID_PATH}')

except ImportError:
    print('xESMF not installed — skipping regrid step.')
    print('Install: conda install -c conda-forge xesmf')

xESMF not installed — skipping regrid step.
Install: conda install -c conda-forge xesmf


## Step 4: Dask Parallel Stats

### Key concept: lazy evaluation

```python
# Without Dask:
ds = xr.open_dataset('huge_era5.nc')  # loads everything into RAM → crash
mean = ds.t850.mean()                 # computes on full array

# With Dask:
ds = xr.open_zarr('era5.zarr')        # loads nothing — just metadata
mean = ds.t850.mean()                 # builds a task graph — nothing computed yet
result = mean.compute()              # NOW it executes, chunk by chunk
```

Build **all** graphs before calling `.compute()` — Dask can then fuse operations
and avoid reading each chunk more than once.

In [41]:
    # 'geopotential', '2m_temperature','u_component_of_wind',
    # 'v_component_of_wind', 'specific_humidity', '2m_dewpoint_temperature',
    # 'volumetric_soil_water_layer_1',
    # 'leaf_area_index_high_vegetation',

# Build all lazy graphs before computing any
temp_mean = (ds_local['2m_temperature'] - 273.15).mean('time')
temp_std = (ds_local['2m_temperature'] - 273.15).std('time')
temp_max = (ds_local['2m_temperature'] - 273.15).max('time')

geopotential_mean  = ds_local['geopotential_500'].mean('time')
geopotential_std  = ds_local['geopotential_500'].std('time')
geopotential_max   = ds_local['geopotential_500'].max('time')

print('Lazy graphs built — computing now ...')
t0 = time.time()

# Compute all three in one pass — Dask reads each chunk once for all ops
temp_mean_val, temp_std_val, temp_max_val = dask.compute(
    temp_mean, temp_std, temp_max
)
elapsed = time.time() - t0

print(f'Computed in {elapsed:.2f}s')
print(f'Mean 2m temp (°C):          {float(temp_mean_val.mean()):.2f}')
print(f'2m temp std:  {float(temp_std_val.mean()):.4f}')
print(f'2m temp max:            {float(temp_max_val.mean()):.4f}')

Lazy graphs built — computing now ...
Computed in 0.09s
Mean 2m temp (°C):          19.57
2m temp std:  4.3316
2m temp max:            28.8085


## Step 5: PyTorch Dataset / DataLoader

### Design decisions

- `__getitem__` uses `isel(time=slice(...))` — loads only `seq_len+1` chunks, not the full dataset
- Normalization stats computed once at init, stored as scalars. In production: compute over a full year and save to JSON
- `num_workers=4, persistent_workers=True`: M2/32GB handles multiprocess workers without OOM (was 0 on M1 due to fork copy-on-write memory pressure)
- `pin_memory=False`: only helps with CUDA, not MPS

In [42]:
class ERA5Dataset(Dataset):
    """
    Zarr-backed sliding window dataset for ConvLSTM training.

    Returns:
        x: (seq_len, C, H, W)  — input sequence
        y: (C, H, W)           — target next frame
    """

    def __init__(self, zarr_path: str, variables: Sequence[str], seq_len: int = 6) -> None:
        self.ds        = xr.open_zarr(zarr_path, consolidated=True)[list(variables)]
        self.variables = list(variables)
        self.seq_len   = seq_len
        self.n_times   = self.ds.sizes['time']

        print('  Computing normalization stats ...')
        self.means: dict[str, float] = {}
        self.stds:  dict[str, float] = {}

        for v in self.variables:
            mean_val = float(self.ds[v].mean().compute())
            std_val  = float(self.ds[v].std().compute())
            self.means[v] = mean_val
            self.stds[v]  = std_val if std_val > 0 else 1.0
            print(f'    {v}: mean={mean_val:.4f}  std={std_val:.4f}')

    def __len__(self) -> int:
        return self.n_times - self.seq_len

    def __getitem__(self, idx: int):
        # isel with slice loads only these chunks — not the full dataset
        window = self.ds.isel(time=slice(idx, idx + self.seq_len + 1))

        arr = np.stack(
            [((window[v].values - self.means[v]) / self.stds[v]).astype(np.float32)
             for v in self.variables],
            axis=1,
        )  # (T+1, C, H, W)

        x = torch.from_numpy(arr[:self.seq_len])   # (seq_len, C, H, W)
        y = torch.from_numpy(arr[self.seq_len])    # (C, H, W)
        return x, y

In [45]:
dataset = ERA5Dataset(
    zarr_path=LOCAL_ZARR_PATH,
    variables=  ['2m_temperature', '2m_dewpoint_temperature', '10m_u_component_of_wind', '10m_v_component_of_wind', 'temperature_850', 'specific_humidity_850', 'geopotential_500'],
    # ['2m_temperature', 'volumetric_soil_water_layer_1', 'leaf_area_index_high_vegetation'],
    seq_len=6,
)

print(f'Dataset length: {len(dataset)} windows')
x, y = dataset[0]
print(f'Sample x: {x.shape}  (seq_len=6, C, H, W)')
print(f'Sample y: {y.shape}  (C, H, W)')
print(f'x mean: {x.mean():.4f}  std: {x.std():.4f}  (should be ~0, ~1)')

loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=0, pin_memory=False)
batch_x, batch_y = next(iter(loader))
print(f'Batch x: {batch_x.shape}  (B=4, T=6, C=3, H, W)')
print(f'Batch y: {batch_y.shape}  (B=4, C=3, H, W)')

  Computing normalization stats ...
    2m_temperature: mean=292.7163  std=5.8306
    2m_dewpoint_temperature: mean=283.2760  std=4.1017
    10m_u_component_of_wind: mean=1.4586  std=2.1267
    10m_v_component_of_wind: mean=0.3230  std=2.6321
    temperature_850: mean=287.7550  std=3.3078
    specific_humidity_850: mean=0.0060  std=0.0022
    geopotential_500: mean=56316.7070  std=557.4507
Dataset length: 114 windows
Sample x: torch.Size([6, 7, 13, 33])  (seq_len=6, C, H, W)
Sample y: torch.Size([7, 13, 33])  (C, H, W)
x mean: -0.1854  std: 1.1497  (should be ~0, ~1)
Batch x: torch.Size([4, 6, 7, 13, 33])  (B=4, T=6, C=3, H, W)
Batch y: torch.Size([4, 7, 13, 33])  (B=4, C=3, H, W)


## Cloud Storage Patterns

The zarr API is identical for local and cloud stores — just swap the path:

```python
# Read public ARCO ERA5 (no credentials)
ds = xr.open_zarr(
    'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3',
    consolidated=True,
    storage_options={'token': 'anon'},
    chunks={'time': 1},
)

# Write to GCS
ds.to_zarr('gs://your-bucket/era5_processed.zarr', mode='w', consolidated=True)

# Write to S3
import s3fs
store = s3fs.S3Map('s3://your-bucket/era5_processed.zarr', s3=s3fs.S3FileSystem())
ds.to_zarr(store, mode='w', consolidated=True)
```

Point `ERA5Dataset` at a cloud zarr to train directly from cloud storage:

```python
dataset = ERA5Dataset(
    zarr_path='gs://your-bucket/era5_processed.zarr',
    variables=['2m_temperature', ...],
)
```

In [47]:
import torch
import torch.nn as nn


class ConvLSTMCell(nn.Module):
    """Single ConvLSTM cell — processes one timestep at one spatial scale."""

    def __init__(self, in_channels, hidden_channels, kernel_size):
        super().__init__()
        self.hidden_channels = hidden_channels
        padding = kernel_size // 2  # same padding — preserves H and W

        # One conv that computes all four gates in a single pass
        self.conv = nn.Conv2d(
            in_channels=in_channels + hidden_channels,  # concat x and h
            out_channels=4 * hidden_channels,           # i, f, o, g stacked
            kernel_size=kernel_size,
            padding=padding,
        )

    def forward(self, x, h, c):
        # x: (N, C_in, H, W)
        # h: (N, hidden, H, W)  — hidden state from previous timestep
        # c: (N, hidden, H, W)  — cell state from previous timestep

        combined = torch.cat([x, h], dim=1)       # (N, C_in + hidden, H, W)
        gates = self.conv(combined)               # (N, 4*hidden, H, W)

        # Split into four gates along the channel dimension
        i, f, o, g = gates.chunk(4, dim=1)

        i = torch.sigmoid(i)   # input gate   — what to write
        f = torch.sigmoid(f)   # forget gate  — what to keep from c
        o = torch.sigmoid(o)   # output gate  — what to expose as h
        g = torch.tanh(g)      # cell gate    — candidate values to write

        c_next = f * c + i * g          # update cell state
        h_next = o * torch.tanh(c_next) # update hidden state

        return h_next, c_next

    def init_hidden(self, batch_size, height, width, device):
        """Zero-initialize h and c for the start of a sequence."""
        return (
            torch.zeros(batch_size, self.hidden_channels, height, width, device=device),
            torch.zeros(batch_size, self.hidden_channels, height, width, device=device),
        )


class ConvLSTMForecast(nn.Module):
    """
    Stack of ConvLSTM layers that consumes a sequence of T frames
    and produces a single forecast frame.

    Input:  (N, T, C, H, W)
    Output: (N, C_out, H, W)
    """

    def __init__(self, in_channels, hidden_channels, out_channels,
                 num_layers=2, kernel_size=3):
        super().__init__()
        self.num_layers = num_layers

        # Build the stack — first layer takes in_channels, rest take hidden_channels
        self.cells = nn.ModuleList()
        for i in range(num_layers):
            cell_in = in_channels if i == 0 else hidden_channels
            self.cells.append(ConvLSTMCell(cell_in, hidden_channels, kernel_size))

        # Projection head: maps final h to forecast
        self.head = nn.Sequential(
            nn.Conv2d(hidden_channels, hidden_channels // 2, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(hidden_channels // 2, out_channels, kernel_size=1),
        )

    def forward(self, x):
        # x: (N, T, C, H, W)
        N, T, C, H, W = x.shape
        device = x.device

        # Initialize h and c to zeros for every layer
        states = [
            cell.init_hidden(N, H, W, device)
            for cell in self.cells
        ]

        # Unroll the loop over T
        for t in range(T):
            inp = x[:, t]          # (N, C, H, W) — strip the T dimension

            for i, cell in enumerate(self.cells):
                h, c = states[i]
                h, c = cell(inp, h, c)
                states[i] = (h, c)
                inp = h            # next layer receives this layer's h as input

        # After the loop, states[last] holds the final h
        final_h = states[-1][0]    # (N, hidden, H, W)

        return self.head(final_h)  # (N, out_channels, H, W)

In [48]:
# 1. Instantiate the model

model = ConvLSTMForecast(
    in_channels=7,
    hidden_channels=64,
    out_channels=7,
    num_layers=2,
    kernel_size=3,
)

# In channels and out channels are 7 to match your C dimension. 
# Hidden channels and num_layers are hyperparameters you'll tune later — 
# 64 and 2 are reasonable starting points.

In [49]:
model.eval()
with torch.no_grad():
    pred = model(batch_x)

print(f'Input:  {batch_x.shape}')   # (4, 6, 7, 13, 33)
print(f'Output: {pred.shape}')       # should be (4, 7, 13, 33)
print(f'Output mean: {pred.mean():.4f}  std: {pred.std():.4f}')

Input:  torch.Size([4, 6, 7, 13, 33])
Output: torch.Size([4, 7, 13, 33])
Output mean: -0.0218  std: 0.1167


In [50]:
criterion = nn.MSELoss()
loss = criterion(pred, batch_y)
print(f'Initial loss: {loss.item():.4f}')

Initial loss: 0.9826


In [52]:
# Forward pass in training mode — no torch.no_grad()
model.train()
pred = model(batch_x)
loss = criterion(pred, batch_y)
print(f'Loss: {loss.item():.4f}')

# Now backward will work
loss.backward()
print('Backward pass: OK')

for name, param in model.named_parameters():
    if param.grad is not None:
        print(f'{name}: grad norm = {param.grad.norm():.4f}')
    else:
        print(f'{name}: NO GRADIENT')

Loss: 0.9826
Backward pass: OK
cells.0.conv.weight: grad norm = 0.0774
cells.0.conv.bias: grad norm = 0.0049
cells.1.conv.weight: grad norm = 0.0940
cells.1.conv.bias: grad norm = 0.0217
head.0.weight: grad norm = 0.0211
head.0.bias: grad norm = 0.0772
head.2.weight: grad norm = 0.0445
head.2.bias: grad norm = 0.1786


In [54]:
# Split 80/20
n = len(dataset)
n_train = int(0.8 * n)
n_val = n - n_train

train_ds, val_ds = torch.utils.data.random_split(dataset, [n_train, n_val])

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, num_workers=0)

pred has shape (4, 7, 13, 33) — for each of the 4 windows in the batch, the model is outputting a predicted value at every grid point (13×33) for each of the 7 variables, one timestep ahead of the input sequence.
So concretely:

Input x: timesteps 0–5, all 7 variables, full spatial grid
Target y: timestep 6, all 7 variables, full spatial grid
pred: the model's guess at timestep 6, all 7 variables, full spatial grid

The MSE loss then computes the squared difference between pred and y at every one of those 7 × 13 × 33 = 3,003 values per sample, averages them, and that's your single loss number.

The physical interpretation:
The model is simultaneously predicting:

What 2m temperature will be at every grid point one timestep ahead
What dewpoint will be
What the u and v wind components will be
What 850 hPa temperature and specific humidity will be
What 500 hPa geopotential will be

All from having watched the previous 6 timesteps of all those variables evolve across the spatial domain.


The thing worth noting for interviews:
The MSE treats all 7 variables equally, but they're on completely different physical scales even after normalization — a 1-unit error in geopotential is not the same physical significance as a 1-unit error in specific humidity. When you get to Phase 2 building the real training pipeline, per-variable loss weighting or reporting per-variable RMSE separately is something to think about.

In [ ]:
train_history = []
val_history   = []

for epoch in range(20):
    # --- train ---
    model.train()
    train_loss = 0.0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        pred = model(batch_x)
        loss = criterion(pred, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # --- validate ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            pred = model(batch_x)
            val_loss += criterion(pred, batch_y).item()

    train_avg = train_loss / len(train_loader)
    val_avg   = val_loss   / len(val_loader)
    train_history.append(train_avg)
    val_history.append(val_avg)

    print(f'Epoch {epoch+1:02d}  train: {train_avg:.4f}  val: {val_avg:.4f}')

In [ ]:
import matplotlib.pyplot as plt

# Get one sample in eval mode
model.eval()
with torch.no_grad():
    sample_x, sample_y = dataset[0]
    pred = model(sample_x.unsqueeze(0))  # (1, C, H, W)

pred_np   = pred.squeeze(0).numpy()    # (C, H, W)
target_np = sample_y.numpy()           # (C, H, W)

variables = ML_VARS
n_vars = len(variables)

fig, axes = plt.subplots(n_vars, 3, figsize=(12, 2.5 * n_vars))
fig.suptitle('ConvLSTM — Predicted vs Target (normalized units)', y=1.01)

for i, var in enumerate(variables):
    vmin = min(pred_np[i].min(), target_np[i].min())
    vmax = max(pred_np[i].max(), target_np[i].max())

    axes[i, 0].imshow(target_np[i], origin='upper', vmin=vmin, vmax=vmax, cmap='RdBu_r')
    axes[i, 0].set_title(f'{var}\nTarget', fontsize=8)
    axes[i, 0].axis('off')

    im = axes[i, 1].imshow(pred_np[i], origin='upper', vmin=vmin, vmax=vmax, cmap='RdBu_r')
    axes[i, 1].set_title('Predicted', fontsize=8)
    axes[i, 1].axis('off')
    fig.colorbar(im, ax=axes[i, 1], fraction=0.046, pad=0.04)

    err = pred_np[i] - target_np[i]
    emax = max(abs(err.min()), abs(err.max()))
    axes[i, 2].imshow(err, origin='upper', vmin=-emax, vmax=emax, cmap='bwr')
    axes[i, 2].set_title('Error (pred − target)', fontsize=8)
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(train_history) + 1)

plt.figure(figsize=(8, 4))
plt.plot(epochs, train_history, label='Train')
plt.plot(epochs, val_history,   label='Val')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('ConvLSTM Training — ERA5 NE MN')
plt.legend()
plt.tight_layout()
plt.show()